<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Week 2 - Incorporating Tools and Memory in Agents</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/5971243/pexels-photo-5971243.jpeg" width="640"/>
</p></center>

<center><font size=6>Reimbursement Automation
</center></font>

## **Problem Statement**

### Business Context

In many mid-sized and large organizations, **employee expense processing and reimbursement** continues to rely heavily on manual, paper-based workflows. Employees submit physical or scanned receipts, which finance teams must individually review, categorize, and validate against company expense policies. This process demands significant time and effort to verify expense details such as category, amount, date, vendor, and compliance requirements.

As organizations grow, the **volume of expense claims increases substantially**, placing additional strain on finance teams. Manual processing creates operational bottlenecks, slows reimbursement cycles, and raises processing costs. It also increases the likelihood of human errors, duplicate reimbursements, and policy violations. Delays in reimbursements can negatively affect employee satisfaction and erode confidence in internal financial processes.

Additionally, expense policy interpretation often varies across reviewers, leading to **inconsistent approval decisions** and increased audit and compliance risks. The lack of automation limits real-time visibility into organizational spending patterns, making it difficult for finance teams to proactively enforce controls, detect anomalies, or identify potential misuse early in the process.


### Objective

The objective is to simplify and streamline the expense processing and reimbursement workflow by leveraging intelligent automation. Routine tasks such as receipt data extraction, expense categorization, and policy validation can be handled automatically, reducing manual effort and processing time. This enables finance teams to focus on exception handling, compliance oversight, and financial analysis rather than repetitive verification tasks.

* Reduce manual effort in reviewing and processing expense claims
* Minimize errors and policy non-compliance in reimbursements
* Accelerate reimbursement turnaround time for employees

### Data Description

This dataset is used to manage and process employee reimbursement requests. It consists of a CSV file that stores incoming reimbursement requests along with a structured reimbursement database containing employee, reimbursement history, and policy information.


**CSV File Attributes**

The CSV file stores sample bills submitted by employees. Whenever a new reimbursement request arrives, it is recorded in this file.

* **Raw Bill (raw_bill)**:
  Textual field containing the raw bill or receipt details (extracted from the image submitted by the employee using OCR).

* **Employee ID (emp_id)**:
  Identifier of the employee who submitted the reimbursement request.

* **Employee Name (emp_name)**:
  Name of the employee associated with the reimbursement request.

**Reimbursement Database Tables**

The reimbursement database consists of three tables.

1. Employee Details Table

    * **Employee ID**: Unique identifier for each employee.

    * **Name**: Name of the employee.

    * **Position**: Job role or designation of the employee.


2. Past Reimbursements Table

    * **ID**: Unique identifier for each reimbursement record.

    * **Employee ID**: Identifier linking the reimbursement record to an employee.

    * **Reimbursement Date**: Date on which the reimbursement was recorded.

    * **Category**: Expense category associated with the reimbursement.

    * **Amount**: Reimbursement amount.


3. Reimbursement Policy Table

    * **Position**: Employee position to which the policy applies.

    * **Category**: Expense category covered by the policy.

    * **Reimbursement Amount**: Maximum allowable reimbursement amount for the given position and category.

## **Installing and Importing Necessary Libraries and Dependencies**

In [2]:
!pip install -q openai==1.66.3 \
                langchain==0.3.20 \
                langchain-openai==0.3.9 \
                langchain-community==0.3.19 \
                langgraph==0.3.21 \
                fastmcp==2.14.4 \
                mcp==1.25.0 \
                nest_asyncio==1.6.0

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
import os
import json
import re
import time
import pandas as pd
import threading
import sqlite3
import asyncio
from typing import TypedDict

import nest_asyncio
nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langgraph.graph import StateGraph, END

from fastmcp import FastMCP
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning,module="jupyter_client")


## **Setting Up Email Credentials and SMTP Configuration**

As part of the reimbursement workflow, the system must notify employees about the outcome of their reimbursement requests, which requires controlled access to an email service for sending notifications.


To enable automated reimbursement notifications, the system needs permission to send emails on behalf of a sender account. This can be achieved using **SMTP credentials**, which allow the application to authenticate with an email service and deliver messages securely.


Instead of hardcoding credentials inside the logic, we externalize all email-related settings into a
configuration file. This makes the notebook:

- Safer (credentials are not scattered in code)
- Easier to maintain
- Portable across email providers (Gmail, Outlook, Zoho, corporate SMTP)

We use an **App Password** instead of a login password to ensure restricted, audit-safe access.


### Generating an App Password (Gmail)

Most email providers do not allow direct login passwords to be used for automated access. Instead, they require an **App Password**, which is a restricted credential specifically designed for programmatic email access.

For Gmail accounts:
1. Enable **2-Step Verification** in your Google account.
2. Navigate to **App Passwords** under Security settings.
3. Create a new app password for *Mail*.
4. Use the generated password for SMTP authentication.

This approach limits access to email functionality only and improves overall security.


**NOTE:** While the steps above are shown for Gmail, most other email providers (such as Outlook, Zoho, and Rediffmail) follow a similar approach and require app-specific or service-specific passwords for secure SMTP access.

### Storing Email Credentials

Only sensitive credentials required for authentication are stored in the configuration file. These include:

* The OpenAI API key and base URL
* The sender email address
* The app password associated with that email account

All other email behavior settings are configured separately to keep the system flexible and easy to update.


```
{
  "OPENAI_API_KEY": "your-openai-key",
  "OPENAI_API_BASE": "your-openai-base",

  "SENDER_EMAIL": "your-email-id",
  "GMAIL_APP_PASSWORD": "your-app-password"
}
```

### SMTP Host and Port Configuration

SMTP host and port define the **email service responsible for delivering messages** from the system.  
These settings must match the provider of the sender email account.

If the sender email belongs to a different service, the SMTP host and port must be updated accordingly to ensure successful authentication and email delivery.


Below are commonly used SMTP configurations. Select the configuration that matches the sender email service.

| Email Provider        | SMTP Host              | Port | Encryption |
|----------------------|------------------------|------|------------|
| Gmail                | smtp.gmail.com         | 465  | SSL        |
| Gmail (STARTTLS)     | smtp.gmail.com         | 587  | STARTTLS  |
| Outlook / Office365  | smtp.office365.com     | 587  | STARTTLS  |
| Yahoo Mail           | smtp.mail.yahoo.com    | 465  | SSL        |
| Zoho Mail            | smtp.zoho.com          | 465  | SSL        |
| Corporate Email      | Provided by IT Team    | Varies | Varies |


In this case study, we use a **Gmail sender account authenticated via an App Password**.  
Accordingly, the SMTP host and port are configured for **Gmail’s SMTP service**.

In [4]:
port=465

SMTP_host='smtp.gmail.com'

Email delivery relies on two things:
1. Valid authentication credentials (sender email and app password)
2. Correct SMTP host and port for the email provider

As long as both are correctly configured, the system can send audit-safe, automated notifications without any changes to the core workflow logic.


## **Data Loading and Model Initialization**


### OpenAI API Calling



In [ ]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")
    GMAIL_APP_PASSWORD = config.get("GMAIL_APP_PASSWORD")
    SENDER_EMAIL = config.get("SENDER_EMAIL")

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

### Data Loading

Let’s start by loading the **`triggered_bill`** dataset, which captures all incoming reimbursement requests submitted by employees. This data serves as the entry point of our workflow, representing the raw claims that will later be validated, analyzed, and processed through business rules and automated decision logic.


In [ ]:
# Loads the triggered_bill.csv file into a Pandas DataFrame, which is required for all downstream analysis
triggered_bill = pd.read_csv("/content/triggered_bill.csv")

Let’s preview the first few reimbursement requests to quickly understand the structure and key fields in the incoming bill data.


In [ ]:
triggered_bill.head()

Let's see a sample bill.

In [ ]:
print(triggered_bill['raw_bill'][1])

## **MCP Setup**

Next, we initialize the MCP instance, which will act as the central control layer for this notebookexposing business tools (like SQL queries and email actions) and enabling structured, tool-based interactions over a controlled interface.


In [ ]:
mcp = FastMCP("demo", host="127.0.0.1", port=8000)

### SQL MCP TOOL

This tool allows MCP to execute SQL queries on the database in a controlled and structured way.


The `@mcp.tool` decorator registers this function as a callable MCP tool, making it safely accessible to the agent through the MCP server instead of allowing direct database access.

In [ ]:
@mcp.tool()
def run_sql(db_path: str, sql: str) -> str:
    """
    Execute SQL on sqlite db and return JSON:
      {"columns":[...], "rows":[...]}
    or {"error":"..."}.
    """
    try:
        # Establish a connection to the SQLite database
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        # Execute the provided SQL query
        cur.execute(sql)

        # Handle read queries (SELECT / WITH)
        if sql.strip().lower().startswith(("select", "with")):
            rows = cur.fetchall()
            cols = [d[0] for d in cur.description] if cur.description else []
            conn.close()
            return json.dumps({"columns": cols, "rows": rows})
        else:
            # Handle write operations (INSERT / UPDATE / DELETE)
            conn.commit()
            conn.close()
            return json.dumps({"status": "ok"})

    except Exception as e:
        # Return any execution error in a structured format
        return json.dumps({"error": str(e)})

### EMAIL MCP Tool

This tool enables MCP to send emails programmatically, allowing the system to trigger real-world notifications and actions as part of an automated workflow.


In [ ]:
@mcp.tool()
def send_email(
    sender_email: str,
    app_password: str,
    receiver_email: str,
    subject: str,
    body: str
) -> str:
    """
    Sends an email using Gmail SMTP.

    This MCP tool is responsible for delivering the final reimbursement
    decision email. It authenticates using the sender's email address
    and a Gmail App Password, then sends the message securely over SSL.

    Returns:
    - {"status": "sent"} on successful delivery
    - {"error": "<message>"} if email sending fails
    """
    # Log the recipient for traceability during execution
    print(f"[INSIDE TOOL]: send_email to={receiver_email}")

    try:
        # Create the email container
        msg = MIMEMultipart()
        msg["From"] = sender_email          # Sender email address
        msg["To"] = receiver_email          # Recipient email address
        msg["Subject"] = subject             # Email subject line

        # Attach the email body as plain text
        msg.attach(MIMEText(body, "plain"))

        # Establish a secure SSL connection to the SMTP server
        # SMTP_host and port are configured earlier for Gmail
        with smtplib.SMTP_SSL(SMTP_host, port) as server:

            # Authenticate using the app password (not the login password)
            server.login(sender_email, app_password)

            # Send the composed email message
            server.send_message(msg)

        # Return a structured success response
        return json.dumps({"status": "sent"})

    except Exception as e:
        # Catch and return any SMTP or authentication errors
        return json.dumps({"error": str(e)})

## **Starting the MCP Server: Making Tools Callable**

Before our system can run SQL queries or send emails, we need a central service that exposes these capabilities in a structured and controlled way.
This is where the MCP (Model Context Protocol) server comes in.

Think of the MCP server as:

- A tool hub that hosts business capabilities (SQL, email, etc.)

- A bridge between our notebook (or an AI agent) and real-world actions

- A controlled execution layer, instead of running arbitrary code directly

To keep the notebook interactive, we start the MCP server in the background.

### Starting the MCP Server in a Background Thread

Here, we configure MCP to run over streamable HTTP, which is required for:

- Long-lived connections

- Tool calls with structured responses

- Two-way communication between client and server

In [ ]:
def start_mcp_server():
    """
    Starts the MCP server using streamable HTTP transport.

    streamable-http allows the MCP server to:
    - Accept HTTP-based streaming connections
    - Support bidirectional communication (read/write streams)
    """
    mcp.run(transport="streamable-http")

We run the server in a daemon thread so:

- The notebook does not freeze

- The server shuts down automatically when the notebook stops

- We can continue executing client-side code in later cells

In [ ]:
# Start the MCP server in a background thread so that
# it does not block the Jupyter notebook execution
server_thread = threading.Thread(
    target=start_mcp_server,
    daemon=True  # Daemon thread exits automatically with the main program
)

server_thread.start()

# Give the server a moment to initialize
time.sleep(2)

print("MCP server started on http://127.0.0.1:8000/mcp")


### Defining How the Client Connects to MCP

Now that the MCP server is running, the client needs to know where to find it.

In [ ]:
# MCP server endpoint
MCP_URL = "http://127.0.0.1:8000/mcp"

This endpoint acts as the single entry point for all tool-based interactions that follow.

### Helper Function: Picking Read & Write Streams


When we connect to the MCP server using a streaming HTTP client, the connection returns multiple internal objects, not all of which are directly useful.

However, MCP communication fundamentally relies on:

- A write stream -> to send requests

- A read stream -> to receive responses

This helper function extracts the correct streams safely.

In [ ]:
def _pick_streams(ctx_tuple):
    """
    streamable_http_client yields a tuple/list with multiple items.
    We pick:
      - write_stream: has .send
      - read_stream: has .receive
    """
    write_stream = None
    read_stream = None

    for obj in ctx_tuple:
        if write_stream is None and hasattr(obj, "send"):
            write_stream = obj
        if read_stream is None and hasattr(obj, "receive"):
            read_stream = obj

    if read_stream is None or write_stream is None:
        raise RuntimeError(
            "Could not find read/write streams in streamable_http_client context. "
            f"Got types: {[type(x) for x in ctx_tuple]}"
        )
    return read_stream, write_stream


This makes the connection logic explicit and robust, instead of relying on assumptions about object order.

### Core MCP Interaction Logic (Async)

This function is the **heart of our MCP-based system**. Its job is to take a tool name (for example, `run_sql` or `send_email`) along with input arguments, connect to the running MCP server, and execute that tool in a safe and structured way.

Conceptually, this function performs four key steps:

1. **Connect to the MCP server** using a streaming HTTP client.
2. **Establish a session** by identifying read/write streams and completing a protocol handshake.
3. **Invoke the requested tool** with the provided arguments.
4. **Parse and return the result** in a clean, predictable format (JSON when possible).

Because MCP communication is asynchronous and stream-based by design, this function is written as `async`. A synchronous wrapper is added later so notebook users can call tools without worrying about async details.


In [ ]:
async def _mcp_tool_call_async(tool_name: str, args: dict) -> dict:
    # Step 1: Open a streaming HTTP connection to the MCP server
    async with streamable_http_client(MCP_URL) as ctx:

        # Ensure the client returns multiple stream-related objects
        # (MCP requires both a read stream and a write stream)
        if not isinstance(ctx, (tuple, list)):
            raise RuntimeError(
                f"Expected tuple/list from streamable_http_client, got {type(ctx)}"
            )

        # Step 2: Extract the correct read and write streams needed for communication
        read_stream, write_stream = _pick_streams(ctx)

        # Step 3: Create a client session using the extracted streams
        async with ClientSession(read_stream, write_stream) as session:

            # Initialize the session to complete the MCP handshake
            # and make registered tools available
            await session.initialize()

            # Step 4: Call the specified MCP tool with the provided arguments
            res = await session.call_tool(tool_name, args)

            # Step 5: Attempt to parse structured text output (usually JSON)
            if res.content and hasattr(res.content[0], "text"):
                txt = res.content[0].text
                try:
                    return json.loads(txt)
                except Exception:
                    # If parsing fails, return the raw text safely
                    return {"raw_text": txt}

            # Final fallback for unexpected response formats
            return {"raw": str(res)}


### Notebook-Safe Wrapper (Why This Exists)


Jupyter notebooks already run an event loop.
Calling async functions directly can cause errors like:

“RuntimeError: event loop is already running”

To avoid this, we expose a synchronous wrapper.

In [ ]:
def mcp_tool_call(tool_name: str, args: dict) -> dict:
    # safe in notebooks because nest_asyncio.apply() is active
    return asyncio.run(_mcp_tool_call_async(tool_name, args))

This allows learners to call MCP tools like normal Python functions.

#### Tool Wrapper: Run SQL via MCP

Finally, we define task-specific helpers that hide protocol details and expose intent clearly.

In [ ]:
def mcp_run_sql(sql: str) -> dict:
    return mcp_tool_call("run_sql", {"db_path": DB_PATH, "sql": sql})

#### Tool Wrapper: Send Email via MCP

This enables real-world actions while keeping credentials and execution logic outside the model.

In [ ]:
def mcp_send_email(to_email: str, subject: str, body: str) -> dict:
    return mcp_tool_call(
        "send_email",
        {
            "sender_email": SENDER_EMAIL,
            "app_password": GMAIL_APP_PASSWORD,
            "receiver_email": to_email,
            "subject": subject,
            "body": body,
        },
    )

## **SQL Toolkit Setup**

In this section, we set up a controlled pipeline that converts a **natural language question** (for example, “What is the total reimbursement by category?”) into a **safe SQLite SQL query**.

Instead of letting the model freely write SQL, we:
- Connect explicitly to a known database
- Expose only approved database tools (tables, schema, query checker)
- Force the model to generate **read-only (SELECT/WITH)** queries
- Validate the generated SQL before execution

This design ensures that the system can *reason about data* while preventing unsafe or destructive database operations.


In [ ]:
# Path to the SQLite database file stored in the notebook environment
DB_PATH = "/content/reimbursement.db"

# SQLAlchemy-compatible database URI for SQLite
DB_URI = f"sqlite:////{DB_PATH.lstrip('/')}"

In [ ]:
# Create a database abstraction that LangChain tools can interact with
db = SQLDatabase.from_uri(DB_URI)

# Initialize a toolkit that provides structured SQL-related tools
# (list tables, inspect schema, validate queries, run queries)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Convert the list of tools into a dictionary for easy access by name
sql_tools = {t.name: t for t in toolkit.get_tools()}

Tools Available in SQL Toolkit

- **sql_db_list_tables**: Retrieves the list of available tables in the connected database, allowing the agent to understand what data sources exist.
- **sql_db_schema**: Fetches the schema details of specified tables, including column names and data types, so queries can be constructed accurately.
- **sql_db_query_checker**: Validates generated SQL queries to ensure they are safe, syntactically correct, and compatible with the database.

In [ ]:
def generate_sql(question: str) -> str:
    """
    Converts a natural language question into a validated SQLite SELECT query.

    This function:
    1. Retrieves available tables
    2. Fetches schema information
    3. Prompts the LLM to generate SQL under strict rules
    4. Validates the SQL before allowing it to be used
    """

    # Step 1: Get the list of available tables in the database
    tables = sql_tools["sql_db_list_tables"].invoke({})

    # Step 2: Fetch schema details for those tables
    schema = sql_tools["sql_db_schema"].invoke({"table_names": tables})

    # Step 3: Construct a tightly controlled prompt for SQL generation
    prompt = f"""
You are a SQL generator. Create ONE sqlite SQL query for the request.

Rules:
- Output SQL only. No markdown. No explanation.
- Use ONLY the tables/columns shown in schema.
- Always quote string literals with single quotes.
- Prefer deterministic results.
- Return rows: category, amount whenever possible.

Tables:
{tables}

Schema:
{schema}

Request:
{question}
"""

    # Step 4: Ask the LLM to generate a SQL query based on the prompt
    sql = llm.invoke(prompt).content.strip()

    try:
        # Step 5: Validate the generated SQL using the query checker tool
        checked = sql_tools["sql_db_query_checker"].invoke({"query": sql})

        # Allow only read-only queries (SELECT or WITH)
        if isinstance(checked, str) and checked.strip().lower().startswith(("select", "with")):
            return checked.strip()

    except Exception:
        # Any validation or parsing failure falls through to rejection
        pass

    # Reject anything that is not a safe, read-only SQL query
    return sql

We are not just generating SQL we are building a guarded reasoning layer where the model can ask questions about data, but only through validated, read-only queries that we explicitly allow.

## **Agentic AI Implementation**

### Define Agent State

This state definition represents the **short-term memory of the agent** during a single reimbursement request.  
It holds all intermediate and final information starting from the raw bill input, moving through structuring, validation, calculation, and finally ending with approval and email notification.

This memory is **temporary and task-scoped**:
- It exists only while processing one reimbursement request
- Each step updates or enriches the same state
- Once the workflow completes, this state can be discarded

In essence, this acts like the agent’s *working notebook*, allowing different reasoning and tool steps to share context without relying on long-term storage.


In [ ]:
class AgentState(TypedDict, total=False):
    Emp_ID: str
    emp_name: str
    raw_bill: str

    structured_bill: str
    date: str
    vendor_details: str
    food_bill_amount: float
    travel_bill_amount: float
    hotel_bill_amount: float
    others_bill_amount: float
    total_bill_amount: float

    structuring_decision: str
    structuring_rationale: str

    max_reimbursement_cap: str
    food_reimbursement_limit: float
    travel_reimbursement_limit: float
    hotel_reimbursement_limit: float

    utilized_cap: str
    food_utilized_amount: float
    travel_utilized_amount: float
    hotel_utilized_amount: float

    food_amount_approved: float
    travel_amount_approved: float
    hotel_amount_approved: float

    calculation_summary: dict
    approved_rationale: str

    mail_subject: str
    mail_text: str

    receiver_email: str
    mail_send_status: dict


### Define Agent Nodes

#### Process Bill Structuring Node



**Bill Structuring Node:** This node converts the raw bill text into a clean, standardized expense breakdown (date, vendor details, category-wise amounts, and total) that can be reliably used for policy checks and reimbursement calculations.


In [ ]:
def process_bill_structuring(state: AgentState):

    structuring_prompt = f"""

### ROLE
You are a senior Corporate expense analyst specializing in vendor-agnostic expense normalization across
hotels, restaurants, and travel agencies.


### INPUT

Raw bill text extracted from receipts or invoices.

Raw Bill Text:
{state["raw_bill"]}


### OBJECTIVE

Extract and compute only the following seven outputs from the raw bill text:

1. Date

   * Invoice or transaction date
   * If unavailable, return `null`

2. Vendor Details (Single String)

   * A single string combining:
     * Vendor name
     * Vendor address
     * Invoice or receipt number
   * Format the string clearly and consistently, for example:
     `"Vendor Name | Address | Invoice #12345"`
   * If vendor details are not identifiable, return `null`

3. Food Bill Amount

   * Total for food, beverages, and liquor + the tax associated with these items(service tax, sales tax, liquor tax) (if applicable)
   * If none present, return `0`

4. Travel Bill Amount

   * Total for flights, taxis, rideshare, rental cars + the tax associated with these items (if applicable)
   * If none present, return `0`

5. Hotel Bill Amount

   * Total for room charges, lodging fees, occupancy taxes + the tax associated with these items (if applicable)
   * If none present, return `0`

6. Others Total Amount

   * Total for parking, tips, laundry, internet, or ambiguous items + the tax associated with these items (if applicable)
   * If none present, return `0`

7. Total Bill amount
   * Final total explicitly stated on the bill that should be equal to the sum of the above fields

### CATEGORIZATION RULES (UNIVERSAL)

* Hotel: room charges, lodging fees, occupancy taxes
* Food: meals, beverages, catering, room service food
* Travel: flights, taxis, rideshare, rental cars, travel agency charges
* Others: parking, tips, laundry, internet, ambiguous items

If classification is ambiguous, assign to Others.


### OUTPUT FORMAT (STRICT  JSON ONLY)

{{
  "date": "YYYY-MM-DD or null",
  "vendor_details": "string or null",
  "food_bill_amount": number,
  "travel_bill_amount": number,
  "hotel_bill_amount": number,
  "others_bill_amount": number
  "total_bill_amount": number
}}


### CONSTRAINTS

* Output must be **valid JSON only**
* No markdown, explanations, or comments
* Do **not** guess missing values
* Use `null` for missing textual data
* Use `0` for missing monetary values
* All monetary values must be tax-inclusive
* Be conservative and audit-safe

"""
    output = llm.invoke(structuring_prompt).content.strip()

    state['structured_bill'] = output

    json_text = re.search(r"\{.*\}", output, re.DOTALL).group()
    result = json.loads(json_text)
    state['date'] = result['date']
    state['vendor_details'] = result['vendor_details']
    state['food_bill_amount'] = float(result['food_bill_amount'])
    state['travel_bill_amount'] = float(result['travel_bill_amount'])
    state['hotel_bill_amount'] = float(result['hotel_bill_amount'])
    state['others_bill_amount'] = float(result['others_bill_amount'])
    state['total_bill_amount'] = float(result['total_bill_amount'])

    return state


#### Structuring Guardrail Node

**Structuring Guardrail Node:** This node acts as a strict audit checkpoint, validating that the structured expense amounts are fully evidence-backed, tax-inclusive, and arithmetically consistent with the original bill before allowing further reimbursement processing.


In [ ]:
def structuring_guardrail(state: AgentState):

  # STRICT expense structuring guardrail prompt

  structuring_guardrail_prompt = f"""
ROLE:
You are a senior expense compliance auditor. Your role is to check the total
for each category from the raw bill text.

INPUTS:
1. Raw bill text (source of truth):
{state["raw_bill"]}

2. Structured expense data to validate (contains only final category totals):
{state["structured_bill"]}

VALIDATION OBJECTIVE:
Approve expense data if all category amounts can be
EXACTLY derived from explicit bill line items and taxes
present in the raw bill.

DETERMINISTIC ACCEPTANCE RULES (ALL MUST PASS):

1. Evidence-based recomposition (CRITICAL):
   - Category totals MAY be computed by summing:
     (base charges + explicitly labeled taxes from the raw bill)
   - Do this reasoning category by category, then produce the answer
   - Arithmetic recomposition is allowed and expected

2. Tax inclusion:
   - All category totals MUST include applicable taxes
   - Taxes are validated only from the raw bill text
   - Structured data is validated only at the final category total level
   - If tax attribution from the raw bill is unclear → BLOCK

3. Category integrity:
   - Food = food + beverage + liquor + liquor tax
   - Hotel = room charges + lodging fees + occupancy taxes
   - Travel = transport charges + transport taxes
   - Other = laundry, parking, and taxes not covered above

4. Arithmetic consistency (STRICT):
   - Each structured category total MUST exactly equal its recomposed total
   - food + travel + hotel + other MUST equal total_bill_amount
   - Exact match required (no tolerance or rounding assumptions)

DECISION RULE:
- APPROVE if ALL rules pass
- BLOCK if ANY rule fails

RATIONALE RULES:
- State the precise validation reason
- If BLOCKED, identify the exact category or numeric mismatch
- If APPROVED, state that totals are deterministically derived and tax-inclusive

OUTPUT FORMAT (STRICT JSON ONLY):
{{
  "decision": "APPROVE" or "BLOCK",
  "rationale": "string"
}}

CONSTRAINTS:
- JSON only
- No corrections or suggestions
- Conservative, arithmetic-first validation
"""


  output = llm.invoke(structuring_guardrail_prompt).content.strip()

  json_text = re.search(r"\{.*\}", output, re.DOTALL).group()
  result = json.loads(json_text)

  state['structuring_decision'] = result['decision']
  state['structuring_rationale'] = result['rationale']

  return state

#### Process Bill Restructuring Node

**Bill Re-structuring Node:** This node repairs only the specific errors flagged by the audit guardrail, using the raw bill as the single source of truth to produce a corrected, tax-inclusive, and arithmetically consistent expense breakdown.


In [ ]:
def process_bill_restructuring(state: AgentState):
    restructuring_prompt = f"""
ROLE:
You are a senior Corporate Expense Analyst specializing in vendor-agnostic expense normalization
across hotels, restaurants, and travel agencies.

You are correcting a previously BLOCKED expense structure.
Your responsibility is to FIX only the identified errors using bill evidence.
You must NOT invent, infer, or enhance data.

INPUTS:
1. Raw bill text (single source of truth):
{state["raw_bill"]}

2. Previously structured expense data (BLOCKED):
{state["structured_bill"]}

3. Auditor rationale explaining why the data was BLOCKED:
{state["structuring_rationale"]}

OBJECTIVE:
Re-structure the expense data by correcting ONLY the issues explicitly mentioned
in the auditor rationale, while keeping all other correct fields unchanged.

You must produce a corrected version of the same structured output schema.

CORRECTION PRINCIPLES (STRICT):

1. Source of truth:
   - The raw bill text is the ONLY authoritative source.
   - Do not rely on assumptions, common patterns, or inferred totals.

2. Scope of correction:
   - Modify ONLY the fields flagged as incorrect in the auditor rationale.
   - Do NOT alter fields that are already correct.

3. Tax handling (MANDATORY):
   - ALL monetary values must be tax-inclusive.
   - If tax is listed separately, add it to the appropriate category:
     - Food → food-related taxes (e.g., liquor tax, service tax on meals, sales tax)
     - Hotel → occupancy tax, lodging tax
     - Travel → transport-related taxes
   - If tax attribution is unclear, include it under Others.

4. Categorization rules (UNIVERSAL):
   - Hotel: room charges, lodging fees, occupancy taxes
   - Food: meals, beverages, liquor, catering, room service food
   - Travel: flights, taxis, rideshare, rental cars, travel agency charges
   - Others: parking, tips, laundry, internet, service fees, ambiguous items
   - If classification is ambiguous, assign to Others.

5. Verification and safety:
   - Do not introduce new vendors, dates, or invoice numbers.
   - Do not change amounts unless they are explicitly incorrect per the rationale.
   - If a value cannot be confidently verified from the bill, use:
     - null for text fields
     - 0 for monetary fields

6. Arithmetic consistency:
   - Category totals must not exceed bill evidence.
   - food + travel + hotel + others MUST equal total_bill_amount.
   - total_bill_amount must exactly match the final total stated on the bill.
   - If the total is missing from the bill, return 0.

OUTPUT FORMAT (STRICT  JSON ONLY):
{{
  "date": "YYYY-MM-DD or null",
  "vendor_details": "string or null",
  "food_bill_amount": number,
  "travel_bill_amount": number,
  "hotel_bill_amount": number,
  "others_bill_amount": number,
  "total_bill_amount": number
}}

CONSTRAINTS:
- Output valid JSON only
- No markdown, explanations, or comments
- Do not guess missing values
- Conservative, audit-safe corrections only
"""

    output = llm.invoke(restructuring_prompt).content.strip()

    state['structured_bill'] = output

    json_text = re.search(r"\{.*\}", output, re.DOTALL).group()
    result = json.loads(json_text)
    state['date'] = result['date']
    state['vendor_details'] = result['vendor_details']
    state['food_bill_amount'] = float(result['food_bill_amount'])
    state['travel_bill_amount'] = float(result['travel_bill_amount'])
    state['hotel_bill_amount'] = float(result['hotel_bill_amount'])
    state['others_bill_amount'] = float(result['others_bill_amount'])
    state['total_bill_amount'] = float(result['total_bill_amount'])

    return state


#### Get Reimbursement Cap Node

This helper function converts raw SQL query results into a clean category-wise map (food, travel, hotel).

In [ ]:
def _rows_to_category_map(result: dict) -> dict:
    out = {"food": 0.0, "travel": 0.0, "hotel": 0.0}
    if not result or "rows" not in result:
        return out
    for row in result["rows"]:
        if len(row) < 2:
            continue
        cat = str(row[0]).strip().lower()
        amt = float(row[1] or 0)
        if cat in out:
            out[cat] = amt
    return out

**Reimbursement Cap Retrieval Node:** This node determines the maximum reimbursable limits for food, travel, and hotel expenses for a specific employee by querying database.


In [ ]:
def get_reimbursement_cap(state: AgentState):
    question = f"""
For employee_id = '{state["Emp_ID"]}', compute the reimbursement cap per category (food, travel, hotel)
based on the employee's position and reimbursement_policy.
Return rows: category, cap_amount
"""
    sql = generate_sql(question)
    mcp_res = mcp_run_sql(sql)

    state["max_reimbursement_cap"] = json.dumps(mcp_res)
    caps = _rows_to_category_map(mcp_res)

    state["food_reimbursement_limit"] = float(caps["food"])
    state["travel_reimbursement_limit"] = float(caps["travel"])
    state["hotel_reimbursement_limit"] = float(caps["hotel"])
    return state

#### Get Previous Reimbursement Details Node

**Previous Utilization Lookup Node:** This node fetches how much reimbursement the employee has already used in each category, providing the necessary context to prevent over-allocation beyond allowed limits.


In [ ]:
def get_previous_reimbursement_details(state: AgentState):
    question = f"""
For employee_id = '{state["Emp_ID"]}', compute total past reimbursed amount per category (food, travel, hotel)
from past_reimbursements.
Return rows: category, total_amount
"""
    sql = generate_sql(question)
    mcp_res = mcp_run_sql(sql)

    state["utilized_cap"] = json.dumps(mcp_res)
    used = _rows_to_category_map(mcp_res)

    state["food_utilized_amount"] = float(used["food"])
    state["travel_utilized_amount"] = float(used["travel"])
    state["hotel_utilized_amount"] = float(used["hotel"])
    return state


#### Calculate Reimbursement

**Reimbursement Calculation Node:** This node applies policy caps and past utilization to compute the final approved reimbursement amounts for each category, ensuring approvals never exceed remaining eligible limits.


In [ ]:
def calculate_reimbursement(state: AgentState):
    food_remaining = max(0.0, state["food_reimbursement_limit"] - state["food_utilized_amount"])
    travel_remaining = max(0.0, state["travel_reimbursement_limit"] - state["travel_utilized_amount"])
    hotel_remaining = max(0.0, state["hotel_reimbursement_limit"] - state["hotel_utilized_amount"])

    state["food_amount_approved"] = min(state["food_bill_amount"], food_remaining)
    state["travel_amount_approved"] = min(state["travel_bill_amount"], travel_remaining)
    state["hotel_amount_approved"] = min(state["hotel_bill_amount"], hotel_remaining)

    state["calculation_summary"] = {
        "food": {
            "requested_amount": state["food_bill_amount"],
            "remaining_cap": food_remaining,
            "approved_amount": state["food_amount_approved"],
        },
        "travel": {
            "requested_amount": state["travel_bill_amount"],
            "remaining_cap": travel_remaining,
            "approved_amount": state["travel_amount_approved"],
        },
        "hotel": {
            "requested_amount": state["hotel_bill_amount"],
            "remaining_cap": hotel_remaining,
            "approved_amount": state["hotel_amount_approved"],
        },
    }
    return state


### Reimbursement Decision Node

**Reimbursement Decision Explanation Node:** This node generates a clear, audit-safe rationale explaining why specific reimbursement amounts were approved, based strictly on requested amounts, remaining caps, and applied policy rules.


In [ ]:
def reimbursement_decision(state: AgentState):
    p = state["calculation_summary"]
    prompt = f"""
ROLE
You explain reimbursement decisions. Do NOT change numbers.

Food:
Requested = {p["food"]["requested_amount"]}
Remaining Cap = {p["food"]["remaining_cap"]}
Approved = {p["food"]["approved_amount"]}

Travel:
Requested = {p["travel"]["requested_amount"]}
Remaining Cap = {p["travel"]["remaining_cap"]}
Approved = {p["travel"]["approved_amount"]}

Hotel:
Requested = {p["hotel"]["requested_amount"]}
Remaining Cap = {p["hotel"]["remaining_cap"]}
Approved = {p["hotel"]["approved_amount"]}

Write an audit-safe rationale ONLY for categories where requested_amount > 0.
"""
    state["approved_rationale"] = llm.invoke(prompt).content.strip()
    return state

### Email Generator

**Email Generation Node:** This node composes a professional, audit-safe reimbursement decision email using the finalized approval details, ensuring consistent and compliant communication with the employee.


In [ ]:
def email_generator(state: AgentState):
    prompt = f"""
ROLE: You are a corporate Finance communication specialist.

TASK: Generate a professional and audit-safe reimbursement decision email.Use the provided inputs exactly as given.

INPUT DATA:
- Employee ID: {state['Emp_ID']}
- Employee Name: {state['emp_name']}
- Transaction Date: {state['date']}
- Vendor Details: {state['vendor_details']}
- Approved Rationale: {state['approved_rationale']}
- Non-reimbursable Amount (Others): {state['others_bill_amount']}
- Total Bill Amount Submitted: {state['total_bill_amount']}

EMAIL REQUIREMENTS:
1. Greet the employee by name
2. Reference transaction date, and vendor details
3. Clearly state the reimbursement decision
4. Include the provided rationale in a neutral, audit-safe manner
5. End with a professional Finance closing

TONE AND CONSTRAINTS:
- Professional, neutral, and factual
- No casual language, no apologies
- No assumptions or recalculations
- Suitable for audit and compliance review

OUTPUT FORMAT (STRICT JSON ONLY):
{{
  "subject": "string",
  "mail_text": "string"
}}

CONSTRAINTS:
- Output valid JSON only
- Do not include markdown, explanations, or extra text
"""
    output = llm.invoke(prompt).content.strip()
    json_text = re.search(r"\{.*\}", output, re.DOTALL).group()
    result = json.loads(json_text)
    state["mail_subject"] = result["subject"]
    state["mail_text"] = result["mail_text"]
    return state

### Email Sender Node

**Email Dispatch Node:** This node sends the finalized reimbursement decision email through MCP, completing the workflow by triggering a real-world notification in a controlled and auditable manner.


In [ ]:
def send_email_via_mcp(state: AgentState):
    to_email = state.get("receiver_email")
    state["mail_send_status"] = mcp_send_email(to_email, state["mail_subject"], state["mail_text"])
    return state


### Build LangGraph Workflow

Add all nodes, define entry point, set conditional edges, and compile the graph into an executable workflow.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("process_bill_structuring", process_bill_structuring)
graph.add_node("structuring_guardrail", structuring_guardrail)
graph.add_node("process_bill_restructuring", process_bill_restructuring)

graph.add_node("get_reimbursement_cap", get_reimbursement_cap)
graph.add_node("get_previous_reimbursement_details", get_previous_reimbursement_details)

graph.add_node("calculate_reimbursement", calculate_reimbursement)
graph.add_node("reimbursement_decision", reimbursement_decision)
graph.add_node("email_generator", email_generator)
graph.add_node("send_email_via_mcp", send_email_via_mcp)

graph.set_entry_point("process_bill_structuring")
graph.add_edge("process_bill_structuring", "structuring_guardrail")

graph.add_conditional_edges(
    "structuring_guardrail",
    lambda state: "approved" if state["structuring_decision"] == "APPROVE" else "restructure",
    {
        "approved": "get_reimbursement_cap",
        "restructure": "process_bill_restructuring",
    },
)

graph.add_edge("process_bill_restructuring", "structuring_guardrail")

graph.add_edge("get_reimbursement_cap", "get_previous_reimbursement_details")
graph.add_edge("get_previous_reimbursement_details", "calculate_reimbursement")
graph.add_edge("calculate_reimbursement", "reimbursement_decision")
graph.add_edge("reimbursement_decision", "email_generator")
graph.add_edge("email_generator", "send_email_via_mcp")
graph.add_edge("send_email_via_mcp", END)

app = graph.compile()

#### Visualize Workflow


Display a visual representation of the node-based workflow using a Mermaid diagram.

In [ ]:
from IPython.display import Image, display

In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

## **Test Cases**

> **NOTE:** Please update the **receiver email address** with a valid sample email ID to test email delivery in this case study. In a real-world system, the employee’s email address would be fetched directly from the database and automatically populated here.


In [ ]:
receiver_email="sample-email"

Through all the test cases we'll explore below, the input to the AI Agent will be a single reimbursement request with the following attributes:

- **Emp_ID**: Unique identifier of the employee submitting the reimbursement  
- **emp_name**: Name of the employee associated with the request  
- **raw_bill**: Raw bill text extracted from the submitted receipt image using OCR  
- **receiver_email**: Email address to which the reimbursement decision notification will be sent  

These inputs initialize the agent's short-term memory and trigger the end-to-end reimbursement processing workflow.


### Test Case 1

In this test case, the employee submits a food reimbursement request that exceeds the available food allowance limit.

In [ ]:
output_1 = app.invoke(
    {
        "Emp_ID": triggered_bill['emp_id'][0],
        "emp_name": triggered_bill['emp_name'][0],
        "raw_bill": triggered_bill['raw_bill'][0],
        "receiver_email": receiver_email,
    }
)

In [ ]:
print(output_1['mail_text'])

**Observation:** The email communicates the reimbursement decision clearly and professionally, providing sufficient context and rationale to ensure transparency and employee understanding.


#### Debug Walkthrough: Test Case 1 (End-to-End Agent Execution)

This section demonstrates how the agent processes a reimbursement request step by step, showing how it transforms raw input into structured outputs and triggers real-world actions.


##### Step 1: Raw Bill Ingestion  


In this step, the agent receives the unstructured bill text extracted from the receipt image and uses it as the starting point for all downstream processing.


In [ ]:
print(triggered_bill['raw_bill'][0])

##### Step 2: Bill Structuring  


In this step, the agent breaks the raw bill text into a structured, standardized format by extracting key fields such as date, vendor details, category-wise amounts, and the total bill amount.


In [ ]:
print(output_1['structured_bill'])

##### Step 3: Guardrail Evaluation  


In this step, the agent checks whether the structured bill is consistent, tax-inclusive, and arithmetically valid before continuing the workflow.


In [ ]:
print(output_1['structuring_decision'])

In [ ]:
print(output_1['structuring_rationale'])

##### Step 4: Policy Retrieval  


In this step, the agent uses the SQL tool to fetch reimbursement limits for the employee based on role and policy rules.


In [ ]:
print(output_1['max_reimbursement_cap'])

##### Step 5: Past Utilization Check  



In this step, the agent retrieves historical reimbursement usage to determine the remaining eligible amount in each category.

In [ ]:
print(output_1['utilized_cap'])

##### Step 6: Reimbursement Calculation  


In this step, the agent computes the final approved reimbursement amounts by applying policy caps to the current expense.


In [ ]:
print(output_1['calculation_summary'])

##### Step 7: Decision Explanation  


In this step, the agent generates an audit-safe explanation describing how the reimbursement decision was reached.


In [ ]:
print(output_1['approved_rationale'])

##### Step 8: Email Generation  


In this step, the agent prepares a professional reimbursement decision email using the finalized outputs.


In [ ]:
print(output_1['mail_subject'])

In [ ]:
print(output_1['mail_text'])

##### Step 9: Email Dispatch  


In this step, the agent sends the reimbursement notification to the employee by invoking the MCP email tool.


In [ ]:
print(output_1['mail_send_status'])

### Test Case 2

In this test case, the employee submits a reimbursement request that includes food and liquor items with applicable taxes (service charge and sales tax), and the total amount exceeds the remaining eligible food cap.

In [ ]:
output_2 = app.invoke(
    {
        "Emp_ID": triggered_bill['emp_id'][1],
        "emp_name": triggered_bill['emp_name'][1],
        "raw_bill": triggered_bill['raw_bill'][1],
        "receiver_email": receiver_email,
    }
)


In [ ]:
print(output_2['mail_text'])

**Observation:** The email conveys the reimbursement outcome clearly and professionally, with a transparent explanation that reinforces policy compliance while maintaining a courteous and employee-friendly tone.


### Test Case 3

In this test case, the employee submits a reimbursement request covering three categories: food, hotel, and others, where the total eligible amount exceeds the remaining allowed limit.

In [ ]:
output_3 = app.invoke(
    {
        "Emp_ID": triggered_bill['emp_id'][2],
        "emp_name": triggered_bill['emp_name'][2],
        "raw_bill": triggered_bill['raw_bill'][2],
        "receiver_email": receiver_email,
    }
)


In [ ]:
print(output_3['mail_text'])

**Observation:** The email provides a clear, well-structured breakdown of category-wise reimbursement decisions, transparently explaining both approved and non-approved amounts while maintaining a professional and respectful tone.


## **Conclusions and Business Recommendations**

- **Adopt MCP-based tool orchestration for controlled automation:**
Use MCP as a centralized execution layer to safely expose business tools (SQL queries and email actions), ensuring AI-driven decisions are executed in a governed and auditable manner.

- **Enable reliable, policy-aware data access using SQL tools:**
Leverage SQL tools through MCP to fetch reimbursement caps, utilization history, and policy data directly from source systems, eliminating manual lookups and reducing data inconsistency.

- **Automate compliant stakeholder communication via Email tools:**
Integrate email tools within MCP to generate and send audit-safe reimbursement decisions automatically, ensuring consistent, timely, and traceable employee communication.

- **Reduce operational errors through tool-level validation:**
Decoupling reasoning from execution ensures only validated SQL queries and approved email content reach execution, minimizing financial and compliance risks.

- **Scale finance operations without increasing overhead:**
By combining MCP with reusable SQL and Email tools, organizations can process higher volumes of reimbursement requests with the same finance team capacity.
